# Тренировка моделей (Colab / Kaggle)

Воспроизведение экспериментов из статьи "Why Can't Transformers Learn Multiplication?"

## 0. Setup

In [1]:
# Клонируем репозиторий (замени URL на свой)
!git clone https://github.com/maximvw/transformer-analyzing.git /content/diplom
%cd /content/diplom

# Установка только недостающих пакетов (torch/transformers уже есть на Kaggle/Colab)
!pip install -q einops fancy-einsum

Cloning into '/content/diplom'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 178 (delta 60), reused 163 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 3.33 MiB | 13.37 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/diplom


In [2]:
# Проверка GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 1. Генерация данных

In [3]:
!gdown https://drive.google.com/uc?id=1Si_mKvQaYJPvaf-3hb3jALOLbhPrEHI6 -O train.txt
!gdown https://drive.google.com/uc?id=1ZI6_Uk_elMB_fRryOxRf06bFpmF5j6HI -O valid.txt
!gdown https://drive.google.com/uc?id=1mZO9g_3xp1vEwJcbYaK9Cej1hk0qldy3 -O test_bigbench.txt

!mv train.txt Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt
!mv valid.txt Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt
!mv test_bigbench.txt Internalize_CoT_Step_by_Step/data/4_by_4_mult/test_bigbench.txt

Downloading...
From (original): https://drive.google.com/uc?id=1Si_mKvQaYJPvaf-3hb3jALOLbhPrEHI6
From (redirected): https://drive.google.com/uc?id=1Si_mKvQaYJPvaf-3hb3jALOLbhPrEHI6&confirm=t&uuid=bef1b5a7-ab6f-422d-a8af-c87202a4e8fa
To: /content/diplom/train.txt
100%|█████████████████████████████████████████| 107M/107M [00:00<00:00, 112MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ZI6_Uk_elMB_fRryOxRf06bFpmF5j6HI
To: /content/diplom/valid.txt
100%|████████████████████████████████████████| 132k/132k [00:00<00:00, 82.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1mZO9g_3xp1vEwJcbYaK9Cej1hk0qldy3
To: /content/diplom/test_bigbench.txt
100%|████████████████████████████████████████| 132k/132k [00:00<00:00, 81.1MB/s]


In [ ]:
# !python generate_data.py

## 2. Тренировка ICoT

2 слоя, 4 головы, n_embd=768 (~54M параметров). Ожидаемо 100% accuracy после ~13 эпох.

In [ ]:
!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/train.py \
    --model gpt2 --n_layer 2 --n_head 4 --n_embd 768 \
    --train_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt \
    --val_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt \
    --lr 5e-5 --batch_size 32 --seed 3456 --reset_optimizer \
    --epochs 1 \
    --remove_per_epoch 8 \
    --remove_all_when_remove_beyond inf \
    --removal_smoothing_lambda 4 \
    --removal_side left \
    --pretrain_epochs 0 \
    --save_model train_models/4_by_4_mult/icot

Namespace(model='gpt2', train_path='Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt', val_path='Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt', test_path=None, epochs=1, lr=5e-05, batch_size=32, accumulate=1, remove_per_epoch=8.0, remove_all_when_remove_beyond=inf, removal_smoothing_lambda=4.0, removal_side='left', pretrain_epochs=0, truncation=-1, max_len_train=-1, max_new_tokens=800, max_size=-1, save_model='train_models/4_by_4_mult/icot', from_pretrained=None, remove_start_from=0, seed=3456, max_grad_norm=1.0, bf16=False, reset_optimizer=True, keep_position=False, reinitialize_weights=False, resume=False, n_layer=2, n_head=4, n_embd=768)
torch.float32 float32 cuda
config.json: 100%|█████████████████████████████| 665/665 [00:00<00:00, 3.81MB/s]
tokenizer_config.json: 100%|██████████████████| 26.0/26.0 [00:00<00:00, 169kB/s]
vocab.json: 1.04MB [00:00, 13.1MB/s]
merges.txt: 456kB [00:00, 7.49MB/s]
tokenizer.json: 1.36MB [00:00, 33.2MB/s]
Creating features from dataset

## 2b. Продолжение тренировки ICoT (Resume)

Если сессия прервалась — продолжить с последнего сохранённого чекпоинта.
Замени `checkpoint_N` на нужный номер эпохи.

In [ ]:
RESUME_FROM = "train_models/4_by_4_mult/icot/checkpoint_0"  # замени 0 на нужный номер

!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/train.py \
    --model gpt2 --n_layer 2 --n_head 4 --n_embd 768 \
    --train_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt \
    --val_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt \
    --lr 5e-5 --batch_size 32 --seed 3456 --reset_optimizer \
    --epochs 200 \
    --remove_per_epoch 8 \
    --remove_all_when_remove_beyond inf \
    --removal_smoothing_lambda 4 \
    --removal_side left \
    --pretrain_epochs 0 \
    --from_pretrained {RESUME_FROM} \
    --resume \
    --save_model train_models/4_by_4_mult/icot

## 3. Тренировка SFT

Без CoT (все токены удалены сразу). Ожидаемо <1% accuracy.

In [ ]:
!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/train.py \
    --model gpt2 --n_layer 2 --n_head 4 --n_embd 768 \
    --train_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/train.txt \
    --val_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/valid.txt \
    --lr 5e-5 --batch_size 32 --seed 3456 --reset_optimizer \
    --epochs 60 \
    --remove_per_epoch 99999 \
    --removal_side left \
    --save_model train_models/4_by_4_mult/sft

## 4. Инференс

In [ ]:
# ICoT inference
!TOKENIZERS_PARALLELISM=false python Internalize_CoT_Step_by_Step/src/generate.py \
    --from_pretrained train_models/4_by_4_mult/icot/checkpoint_12 \
    --test_path Internalize_CoT_Step_by_Step/data/4_by_4_mult/test_bigbench.txt \
    --max_new_tokens 800

## 5. Скачать чекпоинты (опционально)

Сохранить результаты на Google Drive, чтобы не потерять при перезапуске Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r train_models /content/drive/MyDrive/diplom_checkpoints/